# test preview

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
preview = pd.read_csv('/Users/janani/Desktop/vehicle/Motor_Vehicle_Register.csv')

In [ ]:
preview.head()

Removing features

In [ ]:
preview.iloc[:10, :]

# Data exploration

In [ ]:
preview['MOTIVE_POWER'].unique()

MOTIVE_POWER	

BODY_TYPE	-one hot

CC_RATING

GROSS_VEHICLE_MASS

VEHICLE_YEAR	

FC_COMBINED

too many misign valeus - poweroutput not taken

In [ ]:
num_zeros = (preview['POWER_RATING'] == 0).sum()
print(num_zeros)


In [ ]:
preview['POWER_RATING'].value_counts()

In [ ]:
preview['BODY_TYPE'].value_counts()

# Data cleaning

In [ ]:
selectvar = ["MOTIVE_POWER","BODY_TYPE","CC_RATING","GROSS_VEHICLE_MASS","VEHICLE_YEAR", "FC_COMBINED"]
df_1 = preview[selectvar]
print(df_1.head())

In [ ]:
df_1['MOTIVE_POWER'].isna().sum()

In [ ]:
df_1[df_1["MOTIVE_POWER"].isna()]["BODY_TYPE"].value_counts()

Removing  trailers rows as they dont consume fuel

In [ ]:
trailers = [
    "DOMESTIC TRAILER",
    "BOAT TRAILER",
    "CARAVAN" ,
    "OTHER COMMERCIAL TRAILER",
    "FLAT-DECK TRAILER",
    "NON HIGHWAY TRAILER-OTHER",
]

df_2 = df_1[~df_1["BODY_TYPE"].isin(trailers)]
# removing the 7 nan values
df_2 = df_2.dropna(subset=["MOTIVE_POWER"])

In [ ]:
df_2['MOTIVE_POWER'].isna().sum()

In [ ]:
df_2['BODY_TYPE'].isna().sum()

In [ ]:
(df_2['CC_RATING' ]== 0).sum()

In [ ]:
(df_2['GROSS_VEHICLE_MASS' ]== 0).sum()

0 as mass is not possible , so removing them

In [ ]:
df_2 = df_2[df_2["GROSS_VEHICLE_MASS"] != 0]

In [ ]:
df_2['VEHICLE_YEAR'].isna().sum()

In [ ]:
df_2['FC_COMBINED'].isna().sum()

In [ ]:
df_2[df_2["FC_COMBINED"].isna()]["BODY_TYPE"].value_counts()

Removing Agriculture related stuff and oly considering light vehicles

In [ ]:
vehicles_not = [
    "TRACTOR",
    "MOBILE MACHINE",
    "AG MACHINE - OTHER",
    "OTHER AGRICULTURCAL",
    "SELF PROPELLED CARAVAN",
    "CAB AND CHASSIS ONLY"
]
df_2 = df_2[~df_2["BODY_TYPE"].isin(vehicles_not)]

In [ ]:
df_2["BODY_TYPE"].unique()

In [ ]:
df_2 = df_2.dropna(subset=["FC_COMBINED"])

In [ ]:
df_2['FC_COMBINED'].isna().sum()

In [ ]:
df_2["GROSS_VEHICLE_MASS"].agg(["min", "max"])

Removing GVM values which are less than 500

In [ ]:
df_2 = df_2[df_2["GROSS_VEHICLE_MASS"] > 500]

In [ ]:
df_2["MOTIVE_POWER"].unique()

Checking for CNG

In [ ]:
df_2[df_2["MOTIVE_POWER"] == "CNG"]["FC_COMBINED"]

In [ ]:
(df_2["MOTIVE_POWER"] == "CNG").sum()

In [ ]:
remove_power = [
    "CNG"
]

df_2 = df_2[~df_2["MOTIVE_POWER"].isin(remove_power)]

Since there is only one cng, removing that vehicle as it not enough to train and predict

In [ ]:
df_2.isna().sum()

_____________________

removing LPG ,as it has very few values

In [ ]:
(df_2["MOTIVE_POWER"] == "LPG").sum()

In [ ]:
df_2 = df_2[df_2["MOTIVE_POWER"] != "LPG"]

_____________________________

We also see that Electric cars and fuel cells have an FC ? , we are removing it

In [ ]:
(df_2["MOTIVE_POWER"] == "ELECTRIC [PETROL EXTENDED]").sum()

removing Electric [petrol extended] as it has very few values

In [ ]:
remove_power = [
    "ELECTRIC",
    "ELECTRIC FUEL CELL HYDROGEN",
    "ELECTRIC FUEL CELL OTHER",
    "ELECTRIC [PETROL EXTENDED]"
]

df_2 = df_2[~df_2["MOTIVE_POWER"].isin(remove_power)]

______________

Merging the extra categories

In [ ]:
fuel_mapping = {
    "PETROL ELECTRIC HYBRID": "PETROL HYBRID",
    "DIESEL ELECTRIC HYBRID": "DIESEL HYBRID"
}

df_2["MOTIVE_POWER"] = df_2["MOTIVE_POWER"].replace(fuel_mapping)

In [ ]:
df_2["MOTIVE_POWER"].unique()

In [ ]:
df_2["MOTIVE_POWER"].value_counts()

In [ ]:
df_2["MOTIVE_POWER"] = df_2["MOTIVE_POWER"].replace(
    "PLUGIN DIESEL HYBRID",
    "DIESEL HYBRID"
)

In [ ]:
df_2["MOTIVE_POWER"].unique()

# Visualisations

In [ ]:
sns.barplot(x="BODY_TYPE", y="FC_COMBINED", data=df_2)
plt.xlabel("BODY_TYPE")
plt.ylabel("FC_COMBINED")
plt.xticks(rotation=90)
plt.show()

Amongst body types cars wihich people use Hatchback has the least fuel consupmtion
Sports car have the highest fuel consumption

In [ ]:
sns.scatterplot(x="CC_RATING", y="FC_COMBINED", data=df_2)
plt.xlabel('CC_RATING')
plt.ylabel('FC_COMBINED')


No signifcant corellation

In [ ]:
sns.barplot(x="MOTIVE_POWER", y="FC_COMBINED", data=df_2)
plt.xlabel("MOTIVE_POWER")
plt.ylabel("FC_COMBINED")
plt.xticks(rotation=90)
plt.show()

 hybrid is more effient than normal for both petrol an hybrid  , LPG has higher because FC LPG contains less energy per litre than petrol, so you need to burn more LPG to produce the same power

In [ ]:
sns.scatterplot(x="GROSS_VEHICLE_MASS", y="FC_COMBINED", data=df_2)
plt.xlabel('GROSS_VEHICLE_MASS')
plt.ylabel('FC_COMBINED')

 # Decompose engineering

Converting year into age

# Splitting test and train data

Taking a smaple cause the data is huge

In [ ]:
df_2 = df_2.sample(n=200000, random_state=42)

In [ ]:
from sklearn.model_selection import train_test_split

X = df_2.drop("FC_COMBINED", axis=1)  # input features
y = df_2["FC_COMBINED"]               # target

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,      # 20% test, 80% train
    random_state=42
)


# Categorical features - One hot encoding

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class FeatureEngineer(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X["VEHICLE_AGE"] = 2026 - X["VEHICLE_YEAR"]
        return X

    def get_feature_names_out(self, input_features=None):
        return list(input_features) + ["VEHICLE_AGE"]

In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.compose import make_column_selector
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder


preprocessing = make_column_transformer(
    (
        FeatureEngineer(),
        make_column_selector(dtype_include=np.number)
    ),
    (
        OneHotEncoder(sparse_output=False, handle_unknown="ignore"),
        make_column_selector(dtype_include=object)
    )
)

X_train_new = preprocessing.fit_transform(X_train)
X_test_new = preprocessing.transform(X_test)


In [ ]:
feature_names = preprocessing.get_feature_names_out()

X_train_new = pd.DataFrame(
    X_train_new,
    columns=feature_names
)

X_test_new = pd.DataFrame(
    X_test_new,
    columns=feature_names
)

# Train ML models

In [ ]:
from sklearn import linear_model
lin_reg = linear_model.LinearRegression()
lin_reg.fit(X_train_new,y_train)
lin_pred = lin_reg.predict(X_test_new)


In [ ]:
from sklearn import ensemble
rf_reg = ensemble.RandomForestRegressor()
rf_reg.fit(X_train_new,y_train)
rf_pred = rf_reg.predict(X_test_new)

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gb = GradientBoostingRegressor(random_state=42)

gb.fit(X_train_new, y_train)

gb_pred = gb.predict(X_test_new)

 K fold cross validation 

linear model

In [ ]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(lin_reg,X_train_new,y_train,scoring = "neg_mean_squared_error",cv = 10)
lin_rmse_scores = np.sqrt(-scores)
print(lin_rmse_scores)
print(lin_rmse_scores.mean(),lin_rmse_scores.std())

In [ ]:
print(X_train_new.shape)

RandomForestRegressor

In [ ]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(rf_reg, X_train_new, y_train,scoring="neg_mean_squared_error", cv=10)
rf_rmse_scores = np.sqrt(-scores)
print(rf_rmse_scores)
print(rf_rmse_scores.mean(), rf_rmse_scores.std())

gradient boosting

In [ ]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(gb, X_train_new, y_train,scoring="neg_mean_squared_error", cv=10)
gb_rmse_scores = np.sqrt(-scores)
print(gb_rmse_scores)
print(gb_rmse_scores.mean(), gb_rmse_scores.std())

Analyisng the most significant variable

In [ ]:
importance = pd.DataFrame(dict([(var,globals()[var].feature_importances_)
for var in ['rf_reg','gb']]),index = X_train_new.columns)
print(importance.sort_values(by='rf_reg', ascending=False))

In [ ]:
pd.Series(lin_reg.coef_,X_train_new.columns).sort_values()

plugin petrol hybrid , petrol hybrid and diesel hybrid decreasees fuel consumption negative value

Best model

1. random forest
2. gradient boosting
3. Linear model

top features
1. featureengineer__CC_RATING                          5.902614e-01  
2. onehotencoder__MOTIVE_POWER_PETRO                   5.352048e-01


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, KFold
param_grid = {
    "n_estimators": [50,100,200],
    "max_features": ["sqrt", "log2"]
}
forest_reg = RandomForestRegressor()
forest_search = GridSearchCV(forest_reg,param_grid,cv = 10,scoring = 'neg_mean_squared_error')
forest_search.fit(X_train_new,y_train)
print( forest_search.best_params_, forest_search.best_score_)


In [ ]:
from sklearn.ensemble import RandomForestRegressor
import pandas as pd

model = RandomForestRegressor(
    n_estimators=50,
    max_features="log2",
    max_depth=20,
    random_state=42,
    n_jobs=-1
)




model.fit(X_train_new, y_train)

y_pred = pd.Series(
    model.predict(X_test_new),
    index=X_test.index,
    name="FC_COMBINED"
)

In [ ]:
predictions = X_test.copy()
predictions["Predicted_FC_COMBINED"] = y_pred

predictions.to_csv("predicted_fuel_consumption.csv", index=True)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R²:", r2)

STreamlit

In [ ]:
import joblib
joblib.dump(model, "fuel_model.pkl")
joblib.dump(preprocessing, "preprocessing.pkl")